# Unit 7 — Putting it all together

Units 1–6 built the parts. Unit 7 bolts them together, and the lesson is that **composition is
where the errors live**: every seam is a place to lose information, add latency, or be silently
wrong.

This notebook is the inline version of `walkthrough.py` — same nine sections, but the plots render
here and every clip is playable.

1. The cascade — three models, one sentence, and what each seam costs
2. Language forcing — Whisper translates into English only, and the "trick" that isn't
3. Error propagation — an ASR mistake is unrecoverable, because stage 2 never sees audio
4. Latency — additive per stage, and the fast cascade cannot pass the hands-on
5. The text bottleneck — two voices in, one voice out
6. The voice assistant — four stages, two of which no longer exist
7. Transcribing a meeting — word timestamps, the merge, and diarization you cannot fake
8. The hands-on — what the assessor actually calls, and the int16 trap
9. The last seam — a vocabulary that deletes what it cannot say

This is the first unit where the failure of one component shows up as a confusing failure of a
*different* one.

Everything runs on CPU. New here: `opus-mt-en-fr`, `opus-mt-en-ar`, `mms-tts-fra`, `mms-tts-ara`
and `LaMini-Flan-T5-248M`; the rest is cached if you ran Units 2–6. A cold machine pulls about
**1.95 GB** into `~/.cache/huggingface`. *Run All* takes **8–13 minutes** warm.

Section 7 **streams** its corpus, so it needs the network on every run, cached or not.
`facebook/mms-lid-126` — the 3.86 GB language classifier the grading Space uses — is deliberately
**not** downloaded. Section 8 says why.

In [ ]:
%matplotlib inline
import time

import numpy as np
import torch
import matplotlib.pyplot as plt
import IPython.display as ipd

ASR_ID   = "openai/whisper-base"          # 290 MB, cached since Unit 5; stage 1 of the cascade
ASR_TINY = "openai/whisper-tiny"          # 151 MB, cached since Unit 2; the section 4 sweep
MT_ID    = "Helsinki-NLP/opus-mt-en-fr"   # 301 MB, 75.1M params; sentencepiece-only tokenizer
MT_ARA   = "Helsinki-NLP/opus-mt-en-ar"   # 307 MB; section 9's second target, and the Space's
TTS_FRA  = "facebook/mms-tts-fra"         # 145 MB, phonemize:false so no espeak-ng needed
TTS_ARA  = "facebook/mms-tts-ara"         # 145 MB; 38 symbols, no digits, NO punctuation
TTS_ENG  = "facebook/mms-tts-eng"         # 145 MB, cached since Unit 6
LLM_ID   = "MBZUAI/LaMini-Flan-T5-248M"   # 990 MB; the Hub's pipeline_tag on this model is WRONG
WAKE_ID  = "MIT/ast-finetuned-speech-commands-v2"   # 342 MB, cached since Unit 4
T5_ID    = "microsoft/speecht5_tts"       # Unit 6's stack; section 5 uses it as the SOURCE voice
VOC_ID   = "microsoft/speecht5_hifigan"
XVECTOR_ID  = "Matthijs/cmu-arctic-xvectors"
DUMMY_ID    = "hf-internal-testing/librispeech_asr_dummy"
DIALECTS_ID = "ylacombe/english_dialects"
DIALECTS_CONFIG = "northern_female"       # 5 real speakers, same accent and sex: the hard case

SAMPLING_RATE = 16_000
WAKE_WORD = "marvin"   # AST id2label[27]; google/speech_commands' own index is 26
SEED = 7               # VITS has a stochastic duration predictor: seed every call
N_SEAM = 6             # sentences scored in the section 3 seam analysis

# Whisper decodes with num_beams=5 by default: ~5x slower on CPU for a barely better transcript.
ASR_EN  = {"task": "transcribe", "language": "english", "num_beams": 1}
ASR_GEN = {"task": "translate", "num_beams": 1}

torch.set_grad_enabled(False)
print("torch", torch.__version__)

In [ ]:
from transformers import MarianMTModel, MarianTokenizer, VitsModel, VitsTokenizer, pipeline

_CACHE = {}

def asr_pipe(model_id=ASR_ID):
    key = f"asr:{model_id}"
    if key not in _CACHE:
        _CACHE[key] = pipeline("automatic-speech-recognition", model=model_id, device=-1)
    return _CACHE[key]

def translator(model_id=MT_ID):
    key = f"mt:{model_id}"
    if key not in _CACHE:
        _CACHE[key] = (MarianTokenizer.from_pretrained(model_id),
                       MarianMTModel.from_pretrained(model_id))
    return _CACHE[key]

def vits(model_id=TTS_FRA):
    key = f"tts:{model_id}"
    if key not in _CACHE:
        _CACHE[key] = (VitsTokenizer.from_pretrained(model_id),
                       VitsModel.from_pretrained(model_id))
    return _CACHE[key]

def translate_text(text, model_id=MT_ID):
    tok, model = translator(model_id)
    out = model.generate(**tok(text, return_tensors="pt"), num_beams=1, max_new_tokens=256)
    return tok.decode(out[0], skip_special_tokens=True)

def synthesise(text, model_id=TTS_FRA, seed=SEED):
    tok, model = vits(model_id)
    torch.manual_seed(seed)          # VITS's duration predictor is stochastic
    return np.asarray(model(**tok(text=text, return_tensors="pt")).waveform,
                      dtype=np.float32).squeeze()

def audio_dict(array):
    """A FRESH dict for every pipeline call.

    AutomaticSpeechRecognitionPipeline.preprocess MUTATES the dict you hand it - it moves
    "array" into "raw" in place - so reusing one dict across two calls makes the SECOND raise
    `ValueError: ... needs to contain a "raw" key`, which reads like your input was malformed
    rather than already consumed.
    """
    return {"array": np.asarray(array), "sampling_rate": SAMPLING_RATE}

In [ ]:
from datasets import load_dataset

# 73 clean LibriSpeech validation clips with human references and speaker ids (~9 MB, cached
# since Unit 5).
ds = load_dataset(DUMMY_ID, "clean", split="validation")
clip = ds[0]
arr = clip["audio"]["array"]
src_secs = len(arr) / SAMPLING_RATE
print(f"row 0 of {len(ds)}: {src_secs:.2f}s")
print(clip["text"])
ipd.Audio(arr, rate=SAMPLING_RATE)

## 1 — The cascade: three models, one sentence, and what each seam costs

The course builds speech-to-speech as **two** stages: Whisper with `task="translate"`, then TTS.
That produces **English, always**. The hands-on requires a non-English target, so this unit uses
**three**: ASR → machine translation → TTS.

The course's own introduction warns that more components means more error propagation and more
latency. Sections 3 and 4 measure both instead of taking its word for it.

In [ ]:
t0 = time.perf_counter()
english = asr_pipe()(audio_dict(arr), generate_kwargs=ASR_EN)["text"].strip()
t_asr = time.perf_counter() - t0

t0 = time.perf_counter()
french = translate_text(english)
t_mt = time.perf_counter() - t0

t0 = time.perf_counter()
wav = synthesise(french)
t_tts = time.perf_counter() - t0

# Run the whole thing a SECOND time, now that every checkpoint is resident. The first pass
# is what a cold process costs, because each stage loaded its own weights inside its own
# timer; the second is what the models actually cost to run.
t0 = time.perf_counter()
asr_pipe()(audio_dict(arr), generate_kwargs=ASR_EN)
w_asr = time.perf_counter() - t0
t0 = time.perf_counter(); translate_text(english); w_mt = time.perf_counter() - t0
t0 = time.perf_counter(); synthesise(french);      w_tts = time.perf_counter() - t0

print(f"source audio : {src_secs:.2f}s")
print(f"1. ASR : {english}")
print(f"2. MT  : {french}")
print(f"3. TTS : {len(wav) / SAMPLING_RATE:.2f}s of French audio")

total, warm = t_asr + t_mt + t_tts, w_asr + w_mt + w_tts
print(f"\n{'':<14}{'ASR':>9}{'MT':>9}{'TTS':>9}{'total':>9}{'RTF':>8}")
print(f"{'first call':<14}{t_asr:9.2f}{t_mt:9.2f}{t_tts:9.2f}{total:9.2f}{total / src_secs:8.2f}")
print(f"{'warm':<14}{w_asr:9.2f}{w_mt:9.2f}{w_tts:9.2f}{warm:9.2f}{warm / src_secs:8.2f}")
print(f"\n{total - warm:.2f}s of that first row was loading weights, not running them.")

stages = ["ASR\nwhisper-base", "MT\nopus-mt-en-fr", "TTS\nmms-tts-fra"]
params = [72.6, 75.1, 36.3]          # millions, as reported by the checkpoints
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
for r, (times, _lab) in enumerate((((t_asr, t_mt, t_tts), "first call"),
                                   ((w_asr, w_mt, w_tts), "warm"))):
    left = 0.0
    for name, secs, colour in zip(stages, times, ["tab:blue", "tab:orange", "tab:green"]):
        ax1.barh(r, secs, left=left, color=colour, height=0.55,
                 label=name.replace("\n", " ") if r == 0 else None)
        if secs > 0.25:
            ax1.text(left + secs / 2, r, f"{secs:.1f}", ha="center", va="center",
                     color="white", fontsize=8)
        left += secs
ax1.axvline(src_secs, color="tab:red", ls="--", lw=2, label=f"audio length {src_secs:.1f}s")
ax1.set(yticks=[0, 1], yticklabels=["first call", "warm"], xlabel="seconds",
        title="Latency is additive, and the first call is mostly loading")
ax1.legend(loc="lower right", fontsize=8)
ax2.bar(stages, params, color=["tab:blue", "tab:orange", "tab:green"])
for i, p in enumerate(params):
    ax2.text(i, p, f"{p:.0f}M", ha="center", va="bottom")
ax2.set(ylabel="parameters (millions)", title=f"{sum(params):.0f}M parameters in total")
plt.tight_layout(); plt.show()

In [ ]:
print("in  (English source)")
ipd.display(ipd.Audio(arr, rate=SAMPLING_RATE))
print("out (French, three models later)")
ipd.display(ipd.Audio(wav, rate=SAMPLING_RATE))

Look at what crosses each seam: stage 1 hands stage 2 a **string**. Not audio, not a lattice, not a
confidence — a string. That is the whole reason error propagation is possible (section 3), and the
reason section 5's result is what it is.

## 2 — Whisper translates **into English only**, and the "trick" that isn't one

The course's speech-to-speech page offers a shortcut: rather than adding a translation model,
"trick" Whisper into X→Y by setting `task="transcribe"` and `language=Y`. It is tempting, because it
removes a whole stage.

Here is what it actually produces.

In [ ]:
import jiwer

combos = [
    ("transcribe, language=english", {"task": "transcribe", "language": "english"}),
    ("transcribe, language=es",      {"task": "transcribe", "language": "spanish"}),
    ("transcribe, language=french",  {"task": "transcribe", "language": "french"}),
    ("translate,  language=es",      {"task": "translate", "language": "spanish"}),
    ("translate   (no language)",    {"task": "translate"}),
]
results = {}
for label, gen in combos:
    results[label] = asr_pipe()(audio_dict(arr),
                                generate_kwargs=dict(gen, num_beams=1))["text"].strip()
    print(f"{label:<30} {results[label]}")
print(f"{'MT hop (the correct way)':<30} {french}")

same = results["translate,  language=es"] == results["translate   (no language)"]
print(f"\ntranslate+language=es identical to plain translate: {same}")
print("-> with task='translate' the language kwarg is SILENTLY IGNORED. Whisper's translation")
print("   objective has exactly one target, and it is English.")

cers = {label: jiwer.cer(french, text) for label, text in results.items()}
print("\nCER against the MT French:")
for label, cer in cers.items():
    print(f"  {label:<30} {cer:.3f}")

fig, ax = plt.subplots(figsize=(11, 4.5))
labels = [k.replace(", ", ",\n") for k in cers]
colours = ["tab:red" if "translate" in k else "tab:orange" for k in cers]
bars = ax.bar(labels, list(cers.values()), color=colours)
for b, v in zip(bars, cers.values()):
    ax.text(b.get_x() + b.get_width() / 2, v, f"{v:.2f}", ha="center", va="bottom")
ax.set(ylabel="CER vs the MT French",
       title="No Whisper setting reaches the MT hop\n"
             "(red = translate task, which only ever emits English)")
plt.tight_layout(); plt.show()

In [ ]:
trick = results["transcribe, language=french"]
print("the 'trick' (transcribe + language=french)")
ipd.display(ipd.Audio(synthesise(trick), rate=SAMPLING_RATE))
print("the MT hop")
ipd.display(ipd.Audio(synthesise(french), rate=SAMPLING_RATE))

The forced-French decode is not a translation; it is a **transcription nudged toward French
spelling**. On this clip "apostle" came back as *le passé* and "gospel" as *gosse-boule*.

Nothing raised. Nothing warned. It just quietly produced worse French — which is the shape of every
failure in this unit.

## 3 — Error propagation: measure every seam, not just the end of the pipe

A cascade hides where the damage happened. The end user hears bad French and blames the TTS.

So measure each seam separately: run the MT hop **twice**, once on what the ASR heard and once on
the human reference. The difference between the two French strings is exactly the damage the ASR
caused.

In [ ]:
rows = []
for i in range(N_SEAM):
    row = ds[i]
    ref = row["text"].lower()
    hyp = asr_pipe()(audio_dict(row["audio"]["array"]),
                     generate_kwargs=ASR_EN)["text"].strip().lower().rstrip(".")
    asr_wer = jiwer.wer(ref, hyp)
    mt_cer = jiwer.cer(translate_text(ref), translate_text(hyp))
    rows.append((asr_wer, mt_cer))
    print(f"clip {i}  ASR WER {asr_wer:.3f}  |  French divergence CER {mt_cer:.3f}")
    if asr_wer > 0:
        print(f"   ref : {ref[:74]}")
        print(f"   hyp : {hyp[:74]}")

asr_wers = [r[0] for r in rows]
mt_cers = [r[1] for r in rows]
print(f"\nmean ASR WER               : {np.mean(asr_wers):.3f}")
print(f"mean French divergence CER : {np.mean(mt_cers):.3f}")

x = np.arange(len(rows))
fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(x - 0.2, asr_wers, 0.4, label="ASR seam (WER vs human reference)", color="tab:blue")
ax.bar(x + 0.2, mt_cers, 0.4, label="MT seam (CER: ASR-French vs gold-French)", color="tab:orange")
ax.axhline(np.mean(asr_wers), color="tab:blue", ls="--", lw=1)
ax.axhline(np.mean(mt_cers), color="tab:orange", ls="--", lw=1)
ax.set(xticks=x, xlabel="clip", ylabel="error rate",
       title="Damage attributed to the seam that caused it")
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

Where the ASR was clean the French is identical; where it slipped, the French **inherited the slip
and made it worse**. Stage 2 cannot recover, because stage 2 never sees the audio — only the string.

That is the price of the extra stage, and it is the thing the course's introduction warns about in
one sentence without measuring.

## 4 — Latency is additive, and the fast cascade cannot pass the hands-on

One clip, each ASR checkpoint in slot 1, both cascades. `whisper-small` (967 MB) is left out so
*Run All* stays reasonable — `walkthrough.py` has a `USE_WHISPER_SMALL` flag that adds it.

**Every checkpoint is loaded before the clock starts.** That is not tidiness. Timed cold, the
whisper-tiny 3-stage row came out at 14.57s against a 2-stage 3.68s on this machine, which prices
the MT hop at about eleven seconds; warm, both rows land near 1.7s and the hop costs under a tenth
of one. A model load in the wrong place buys you ten seconds of imaginary machine translation.

In [ ]:
row = ds[0]
secs = len(row["audio"]["array"]) / SAMPLING_RATE
ref = row["text"].lower()

# Warm everything this cell will time.
for model_id in (ASR_TINY, ASR_ID):
    asr_pipe(model_id)
translator(); vits(TTS_FRA); vits(TTS_ENG)

print(f"one clip ({secs:.1f}s), both cascades\n")
print(f"{'model':<16}{'ASR s':>7}{'WER':>8}{'3-stage s':>11}{'2-stage s':>11}")
points, deltas = [], []
for model_id in (ASR_TINY, ASR_ID):
    t0 = time.perf_counter()
    hyp = asr_pipe(model_id)(audio_dict(row["audio"]["array"]),
                             generate_kwargs=ASR_EN)["text"].strip().lower().rstrip(".")
    t_asr_m = time.perf_counter() - t0
    wer = jiwer.wer(ref, hyp)

    t0 = time.perf_counter()
    synthesise(translate_text(hyp))
    t_rest3 = time.perf_counter() - t0

    t0 = time.perf_counter()
    en = asr_pipe(model_id)(audio_dict(row["audio"]["array"]),
                            generate_kwargs=ASR_GEN)["text"].strip()
    synthesise(en, TTS_ENG)
    t_two = time.perf_counter() - t0

    name = model_id.split("/")[-1]
    print(f"{name:<16}{t_asr_m:7.2f}{wer:8.3f}{t_asr_m + t_rest3:11.2f}{t_two:11.2f}")
    points.append((name, (t_asr_m + t_rest3) / secs, wer))
    deltas.append((t_asr_m + t_rest3) - t_two)

print(f"\nthe third stage costs {np.mean(deltas):+.2f}s on average here")

fig, ax = plt.subplots(figsize=(9, 4.5))
for name, rtf, wer in points:
    ax.scatter(rtf, wer, s=90)
    ax.annotate(name, (rtf, wer), textcoords="offset points", xytext=(6, 4), fontsize=9)
ax.set(xlabel="real-time factor of the 3-stage cascade (lower is faster)",
       ylabel="ASR WER (lower is better)",
       title="Pick your stage-1 checkpoint\n(bottom-left is what you want)")
plt.tight_layout(); plt.show()

The 2-stage cascade is the cheaper one, and it produces **English** — which is exactly what the
hands-on forbids. You are not choosing between fast and slow. You are paying a fraction of a second
for the only output that can pass.

## 5 — What the text bottleneck destroys: two voices in, one voice out

Unit 6 showed that a 512-float x-vector decides whose voice SpeechT5 uses. Feed two clearly
different voices into this cascade and see what comes out the far end.

This is the heaviest part of the notebook: it loads Unit 6's whole stack (`speecht5_tts` 585 MB +
`speecht5_hifigan` 51 MB + the CMU ARCTIC x-vectors 18 MB) purely to *manufacture two source
voices*. All three are cached if you ran Unit 6.

In [ ]:
import librosa
from transformers import SpeechT5ForTextToSpeech, SpeechT5HifiGan, SpeechT5Processor

proc = SpeechT5Processor.from_pretrained(T5_ID)
t5 = SpeechT5ForTextToSpeech.from_pretrained(T5_ID).eval()
voc = SpeechT5HifiGan.from_pretrained(VOC_ID)
xv = load_dataset(XVECTOR_ID, split="validation")
names = xv["filename"]
print(len(xv), "x-vectors of", len(xv[0]["xvector"]), "dims")

In [ ]:
SENTENCE = "the sun provides energy for life on earth"

outputs = {}
for voice in ("slt", "bdl"):
    idx = next(i for i, f in enumerate(names) if f.startswith(f"cmu_us_{voice}_"))
    vec = torch.tensor(xv[idx]["xvector"]).unsqueeze(0)
    ids = proc(text=SENTENCE, return_tensors="pt")["input_ids"]
    torch.manual_seed(SEED)              # SpeechT5 applies dropout at INFERENCE (Unit 6 §4)
    src = np.asarray(t5.generate_speech(ids, vec, vocoder=voc), dtype=np.float32)
    f0 = float(np.median(librosa.yin(src, fmin=60, fmax=400, sr=SAMPLING_RATE)))

    heard = asr_pipe()(audio_dict(src), generate_kwargs=ASR_EN)["text"].strip()
    fr = translate_text(heard)
    out = synthesise(fr)
    outputs[voice] = (src, f0, heard, fr, out)
    print(f"{voice}: source f0 {f0:6.1f} Hz, {len(src) / SAMPLING_RATE:.2f}s")
    print(f"   heard : {heard}")
    print(f"   french: {fr}")

a, b = outputs["slt"][4], outputs["bdl"][4]
identical = a.shape == b.shape and float(np.max(np.abs(a - b))) == 0.0
print(f"\nsource f0 differ by {abs(outputs['slt'][1] - outputs['bdl'][1]):.1f} Hz")
print(f"output waveforms identical: {identical}  "
      f"(max abs diff {float(np.max(np.abs(a - b))) if a.shape == b.shape else float('nan'):.3e})")

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 5))
for voice, style in (("slt", "-"), ("bdl", "--")):
    src = outputs[voice][0]
    f0 = librosa.yin(src, fmin=60, fmax=400, sr=SAMPLING_RATE)
    ax1.plot(np.linspace(0, len(src) / SAMPLING_RATE, len(f0)), f0, style,
             label=f"{voice} ({outputs[voice][1]:.0f} Hz)")
ax1.set(ylabel="f0 (Hz)", title="Two voices in - clearly different")
ax1.legend(fontsize=8)
for voice, style, w in (("slt", "-", 2.5), ("bdl", "--", 1.2)):
    out = outputs[voice][4]
    f0 = librosa.yin(out, fmin=60, fmax=400, sr=SAMPLING_RATE)
    ax2.plot(np.linspace(0, len(out) / SAMPLING_RATE, len(f0)), f0, style, lw=w, label=voice)
ax2.set(xlabel="seconds", ylabel="f0 (Hz)",
        title="One voice out - the two curves are superimposed")
ax2.legend(fontsize=8)
plt.tight_layout(); plt.show()

for voice in ("slt", "bdl"):
    print(f"{voice} in")
    ipd.display(ipd.Audio(outputs[voice][0], rate=SAMPLING_RATE))
for voice in ("slt", "bdl"):
    print(f"{voice} out")
    ipd.display(ipd.Audio(outputs[voice][4], rate=SAMPLING_RATE))

Everything about *how* the sentence was said — pitch, pace, speaker — died at the first seam,
because **a string cannot carry it**. The two output clips are not merely similar: they are
bit-identical, `max abs diff 0.000e+00`, from sources 78 Hz apart in median f0.

This is the argument for direct speech-to-speech models (Translatotron, in the supplemental reading
below), which skip the text bottleneck entirely. None of them run on a CPU in a few seconds, which
is why the cascade is still what you build.

## 6 — The voice assistant's four stages, and the two that no longer exist

Wake word → transcribe → ask a language model → speak the answer. **Two of the four stages in the
course are no longer runnable, and neither failure is obvious.**

Stage 1 is synthesised here rather than spoken, so the notebook needs no microphone.

In [ ]:
clf = pipeline("audio-classification", model=WAKE_ID, device=-1)
scores, correct, false_wakes = {}, 0, 0
for word in (WAKE_WORD, "stop", "yes", "house"):
    w = synthesise(word, TTS_ENG)
    top = clf(audio_dict(w), top_k=1)[0]
    scores[word] = (top["label"], top["score"])
    if word != WAKE_WORD:
        correct += top["label"] == word
        false_wakes += top["label"] == WAKE_WORD and top["score"] > 0.5
    hit = "WAKE" if top["label"] == WAKE_WORD and top["score"] > 0.5 else "    "
    print(f"{hit} {word:<8} -> {top['label']:<12} {top['score']:.3f}")

print(f"\ncontrols classified correctly : {correct} of 3")
print(f"controls that falsely woke it : {false_wakes} of 3")
print(f"\nthe AST label set has {len(clf.model.config.id2label)} classes and "
      f"'{WAKE_WORD}' is id {clf.model.config.label2id[WAKE_WORD]}.")
print("google/speech_commands puts it at 26 of 36. Compare label STRINGS, never ints.")

fig, ax = plt.subplots(figsize=(10, 4))
words = list(scores)
bars = ax.bar(words, [scores[w][1] for w in words],
              color=["tab:green" if scores[w][0] == WAKE_WORD else "tab:grey" for w in words])
for b, w in zip(bars, words):
    ax.text(b.get_x() + b.get_width() / 2, scores[w][1], scores[w][0],
            ha="center", va="bottom", fontsize=8)
ax.axhline(0.5, color="tab:red", ls="--", label="wake threshold")
ax.set(ylabel="top-1 score", ylim=(0, 1.15),
       title="Wake-word detection on synthesised words\n(green = classified as the wake word)")
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

Read that honestly rather than reporting only the number that flatters it. The wake word scores
0.997, and **none of the three controls is classified correctly** — they simply fail in the safe
direction, as some *other* command rather than as the wake word. So this is not evidence that AST
handles synthesised speech well. It is evidence that this one word lands, which is all a wake word
has to do.

### Stage 3: the language model

The course calls `tiiuae/falcon-7b-instruct` over `api-inference.huggingface.co` — a host whose DNS
record no longer resolves, and a model whose `inferenceProviderMapping` is now empty. So run a small
one locally.

And now the quietest bug in the unit.

In [ ]:
import logging

question = "Answer in one short sentence: what is the capital of France?"

# transformers logs every supported architecture here: a single 3,273-character line listing
# 200+ class names. The complaint is real and is exactly the point, but printing it in full
# buries the lesson. It is logged at ERROR level, not WARNING, so logging.disable() has to be
# told ERROR or it sails straight through.
#
# try/finally is the notebook-only part: in a script a raise here just ends the process, but in
# a kernel it would leave ERROR logging disabled for every later cell.
try:
    logging.disable(logging.ERROR)
    wrong = pipeline("text-generation", model=LLM_ID, device=-1)
    echoed = wrong(question, max_new_tokens=40)[0]["generated_text"].strip()
finally:
    logging.disable(logging.NOTSET)
del wrong                     # ~1 GB of weights we only needed once

right = pipeline("text2text-generation", model=LLM_ID, device=-1)
t0 = time.perf_counter()
answer = right(question, max_new_tokens=40)[0]["generated_text"].strip()
t_llm = time.perf_counter() - t0

print(f"pipeline('text-generation', ...)      -> {echoed!r}")
print(f"pipeline('text2text-generation', ...) -> {answer!r}   ({t_llm:.2f}s)")
print(f"\nechoed the prompt back verbatim: {echoed.strip() == question.strip()}")

The Hub's `pipeline_tag` for `MBZUAI/LaMini-Flan-T5-248M` says `text-generation`, which is **wrong**
— it is a T5, a seq2seq. Pass the task explicitly or your assistant is a parrot that never raises.

`transformers` does complain (*"T5ForConditionalGeneration is not supported for text-generation"*),
muted above because the message lists 200+ class names in one 3,273-character line. **An error you
have to scroll past is an error you stop reading.** And note it is logged at `ERROR`, not `WARNING`,
so `logging.disable(logging.WARNING)` sails straight through it.

In [ ]:
reply = synthesise(answer, TTS_ENG)
print(f"stage 4, spoken: {answer!r}")
ipd.Audio(reply, rate=SAMPLING_RATE)

> Also dead: `from transformers import HfAgent`, which the course's closing section uses to
> generalise the assistant into a tool-using agent. Agents were **removed from transformers
> entirely** — the import raises, it does not deprecate.

`assistant.py` runs these four stages end to end (canned by default, `--live` for a Gradio
push-to-talk front end). It does **not** use `ffmpeg_microphone_live`: that helper's
`_get_microphone_name()` takes device `[0]` from ffmpeg's device list, which on this machine is a
Voicemeeter loopback, and the course's own snippet reads its loop variable from outside the loop.

## 7 — Transcribing a meeting: word timestamps, the merge, and diarization you cannot fake

Two jobs: work out **what** was said and **who** said it, then align them. Whisper does the first
with word timestamps. The second needs a speaker-embedding model, and that is where this stack runs
out of road — so measure exactly how far short it falls.

The LibriSpeech dummy is a single speaker across all 73 rows, so it cannot make a two-speaker track.
The cell below **streams** Unit 6's dialect corpus instead: five real speakers, same accent, same
sex — the hard case, which is what makes the number below credible. It range-reads one ~59 MB
parquet row group, and it needs the network **on every run**, because streaming caches nothing.

In [ ]:
from datasets import Audio

stream = load_dataset(DIALECTS_ID, DIALECTS_CONFIG, split="train", streaming=True)
stream = stream.cast_column("audio", Audio(sampling_rate=SAMPLING_RATE))   # source is 48 kHz

per_speaker = {}
for _, row in zip(range(60), stream):
    per_speaker.setdefault(row["speaker_id"], []).append(row["audio"]["array"])
pair = sorted(per_speaker, key=lambda s: -len(per_speaker[s]))[:2]

turns, truth, cursor = [], [], 0.0
for t in range(4):
    sid = pair[t % 2]
    turn = np.asarray(per_speaker[sid][t // 2], dtype=np.float32)
    turns.append(turn)
    truth.append((cursor, cursor + len(turn) / SAMPLING_RATE, sid))
    cursor += len(turn) / SAMPLING_RATE
meeting = np.concatenate(turns)

print(f"{len(meeting) / SAMPLING_RATE:.1f}s meeting from speakers {pair}, {len(truth)} turns, "
      f"boundaries exact by construction")
ipd.Audio(meeting, rate=SAMPLING_RATE)

In [ ]:
def merge_speakers(segments, chunks, total_s):
    """The speechbox ASRDiarizationPipeline merge, reimplemented with three guards.

    speechbox aligns ONLY on segment end times and consumes the transcript greedily, which is
    fine on well-behaved input and raises on everything else. The guards below are the three
    cases it does not handle: an emptied timestamp array (argmin of an empty sequence), a None
    end timestamp (Whisper emits one for a final unterminated chunk), and a single segment,
    where its merge loop body never runs at all.
    """
    transcript = list(chunks)
    ends = np.array([c["timestamp"][-1]
                     if c["timestamp"] and c["timestamp"][-1] is not None else total_s
                     for c in transcript], dtype=float)
    out = []
    for start, end, sid in segments:
        if ends.size == 0:
            break
        upto = int(np.argmin(np.abs(ends - end)))
        out.append({"speaker": sid, "start": start, "end": end,
                    "text": "".join(c["text"] for c in transcript[: upto + 1]).strip()})
        transcript = transcript[upto + 1:]
        ends = ends[upto + 1:]
    return out

total_s = len(meeting) / SAMPLING_RATE
extra = {"chunk_length_s": 30, "batch_size": 1, "ignore_warning": True} if total_s > 30 else {}
words = asr_pipe()(audio_dict(meeting), generate_kwargs=ASR_EN,
                   return_timestamps="word", **extra)
chunks = words["chunks"]
print(f"whisper word timestamps: {len(chunks)} chunks, "
      f"first {chunks[0]['text']!r} at {chunks[0]['timestamp']}")

print("\nthe merge, against the TRUE boundaries:")
for seg in merge_speakers(truth, chunks, total_s):
    print(f"  [{seg['speaker']}] ({seg['start']:.1f}-{seg['end']:.1f}s) {seg['text'][:56]}")

In [ ]:
from sklearn.cluster import AgglomerativeClustering

def mfcc_diarization_accuracy(audio, truth, win_s=1.0):
    """Cluster 1-second windows by MFCC mean. Returns (accuracy, n_windows)."""
    step = int(win_s * SAMPLING_RATE)
    feats, labels = [], []
    for start in range(0, len(audio) - step, step):
        centre = (start + step / 2) / SAMPLING_RATE
        who = next((sid for s, e, sid in truth if s <= centre < e), None)
        if who is None:
            continue
        mfcc = librosa.feature.mfcc(y=audio[start:start + step], sr=SAMPLING_RATE, n_mfcc=20)
        feats.append(mfcc.mean(axis=1))
        labels.append(who)
    if len(set(labels)) < 2:
        return float("nan"), len(labels)
    pred = AgglomerativeClustering(n_clusters=2, metric="cosine",
                                   linkage="average").fit_predict(np.array(feats))
    truth_arr = np.array([sorted(set(labels)).index(l) for l in labels])
    # cluster labels are arbitrary, so take the better of the two assignments
    return float(max((pred == truth_arr).mean(), (pred != truth_arr).mean())), len(labels)

def chance_baseline(n_windows, trials=20_000, seed=SEED):
    """What the scorer above returns when the clustering is RANDOM. It is NOT 50%.

    Because that max() picks the better of the two cluster-to-speaker mappings, it can never
    fall below 0.5, and on a short track it sits well above it on luck alone: about
    0.5 + sqrt(2 / (pi * n)) / 2, which is 59% at n=19 and still 54% at n=100.
    """
    rng = np.random.default_rng(seed)
    truth_arr = np.zeros(n_windows, dtype=int)
    truth_arr[n_windows // 2:] = 1
    agree = (rng.integers(0, 2, size=(trials, n_windows)) == truth_arr).mean(axis=1)
    return float(np.maximum(agree, 1.0 - agree).mean())

acc, n_win = mfcc_diarization_accuracy(meeting, truth)
chance = chance_baseline(n_win)
print(f"frame accuracy on this track : {acc:.1%}  ({n_win} one-second windows)")
print(f"RANDOM clustering scores     : {chance:.1%}  (measured, 20,000 trials)")
print(f"this method beats chance by  : {acc - chance:+.1%}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(["this track"], [acc * 100], color="tab:red" if acc < chance else "tab:green")
ax.axhline(50, color="0.75", ls=":", label="the 50% everyone quotes")
ax.axhline(chance * 100, color="tab:red", ls=":",
           label=f"RANDOM clustering, measured ({chance:.0%} at n={n_win})")
ax.axhline(80, color="tab:green", ls="--", label="usable threshold")
ax.text(0, acc * 100, f"{acc:.1%}", ha="center", va="bottom")
ax.set(ylabel="frame accuracy (%)", ylim=(0, 105),
       title="MFCC + AgglomerativeClustering is not diarization\n"
             "(the null is not 50%: the scorer picks the better label mapping)")
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

**The null is not 50%.** The scorer takes `max()` over the two cluster-to-speaker mappings, because
cluster ids are arbitrary, so it cannot report below 50% and on a short track it lands near 58% on
luck alone. Quoted against 50% this looks like a weak positive result; against its own measured null
it is noise.

The course does this part with **pyannote**, which is gated (two repo licences and a logged-in
token) and pulls in `torchaudio`, which this repo deliberately does not install. MFCCs encode *what*
was said far more strongly than *who* said it, so without a speaker-embedding model you are
clustering phonetics.

`meeting.py --sweep` scores all ten speaker pairs this way. Measured there: mean 60.3% against a
mean null of 57.8%, range 51.7% to 68.2%, 7 of 10 pairs beat their own null, and **0 of 10 reach a
usable 80%**. Treat this as a measured negative result, not a diarizer.

## 8 — The hands-on contract: what the assessor actually calls

The hands-on asks you to duplicate the course's Space and make it translate into a **non-English**
language. The grader does not read your code. It does this:

```python
client = Client(repo_id)
audio_file = client.predict("test_short.wav", api_name="/predict")
# then facebook/mms-lid-126 on the returned audio, and it fails you if the top label is
# 'eng' or its score is below 0.5
```

Two consequences most people miss:

- **One positional argument in, one `sf.read()`-able return value out.** Add a language dropdown or
  a second output and you fail with a `gradio_client` traceback rather than a grading message.
- The check is only *"not English, confidently"*. It never verifies you hit the language you
  claimed.

In [ ]:
x = np.array([0.9, 1.0, 1.05, 1.5, -1.2], dtype=np.float32)
naive = (x * 32767).astype(np.int16)
clipped = (np.clip(x, -1.0, 1.0) * 32767).astype(np.int16)
print("the template's cast, (x * 32767).astype(np.int16):")
print(f"  float  : {x.tolist()}")
print(f"  naive  : {naive.tolist()}   <- 1.05 became NEGATIVE")
print(f"  clipped: {clipped.tolist()}")

peak = float(np.max(np.abs(wav)))
print(f"\nour French output peaks at {peak:.3f}, so this does not fire today - but it is one")
print("np.clip from being safe, and a louder checkpoint would wrap the sign silently.")

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(13, 3.6))
ax1.stem(x); ax1.axhline(1.0, color="tab:red", ls="--")
ax1.set(title="float samples", ylabel="amplitude")
ax2.stem(naive); ax2.axhline(0, color="0.5", lw=0.8)
ax2.set(title="(x * 32767).astype(int16)\nsign wraps above 1.0")
ax3.stem(clipped); ax3.axhline(0, color="0.5", lw=0.8)
ax3.set(title="np.clip first\nsaturates correctly")
plt.tight_layout(); plt.show()

In [ ]:
out16 = (np.clip(wav, -1.0, 1.0) * 32767).astype(np.int16)
print(f"the exact bytes the grader would consume: {out16.dtype}, {out16.shape}, "
      f"returned as (16000, array)")

# ipd.Audio normalises float input by default and refuses an int16 array when normalize=False,
# so play the round trip back as float. The printed values above are what matters: your ears
# cannot audit this cast, only the numbers can.
ipd.Audio(out16.astype(np.float32) / 32767.0, rate=SAMPLING_RATE)

## 9 — The last seam: a vocabulary that deletes what it cannot say

Section 1 said stage 1 hands stage 2 a string. Stage 3 has the same problem in reverse, and it is
worse, because VITS is character-level and its vocabulary is tiny. `VitsTokenizer` is built with
`normalize=True`, and that path ends in this line of
`transformers/models/vits/tokenization_vits.py`:

```python
filtered_text = "".join(list(filter(lambda char: char in self.encoder, filtered_text)))
```

Not `<unk>`. **Deleted.** Unit 6's SpeechT5 at least emits an `<unk>` you can count; here the
character is dropped from the string before the model ever sees it, nothing is logged, and the audio
comes out fluent and complete.

In [ ]:
# ".,!?;:" plus the Arabic comma, question mark and semicolon.
SENTENCE_PUNCT = ".,!?;:" + "\u060c\u061f\u061b"

arabic = translate_text(english, MT_ARA)
print("the same English sentence, two targets:")
print(f"  en -> fr : {french}")
print(f"  en -> ar : {arabic}")

vocab_rows = []
for label, tts_id, text in (("English", TTS_ENG, english),
                            ("French", TTS_FRA, french),
                            ("Arabic", TTS_ARA, arabic)):
    tok, _ = vits(tts_id)
    # get_vocab() folds in the added special tokens (<unk>, <pad>), which are not symbols the
    # model can pronounce. Count only what it can actually say.
    symbols = sorted(set(tok.get_vocab()) - set(tok.all_special_tokens))
    letters = [c for c in symbols if c.isalpha()]
    digits = [c for c in symbols if c.isdigit()]
    sentence = [c for c in SENTENCE_PUNCT if c in symbols]
    normalised = tok.normalize_text(text)
    filtered, _ = tok.prepare_for_tokenization(text)
    dropped = sorted(set(normalised) - set(filtered))
    vocab_rows.append({"label": label, "symbols": len(symbols), "letters": len(letters),
                       "digits": digits, "sentence": sentence,
                       "n_dropped": len(normalised) - len(filtered), "dropped": dropped})
    print(f"\n{tts_id}")
    print(f"  pronounceable symbols : {len(symbols)}  ({len(letters)} of them letters)")
    print(f"  digits                : {len(digits)}  {''.join(digits) or '(none)'}")
    print(f"  sentence punctuation  : {len(sentence)}  {''.join(sentence) or '(NONE)'}")
    print(f"  deleted from our text : {len(normalised) - len(filtered)} characters  "
          f"{''.join(dropped) or '(none)'}")

no_punct = [r["label"] for r in vocab_rows if not r["sentence"]]
print(f"\nvoices with NO sentence punctuation at all: {', '.join(no_punct)}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
names_v = [r["label"] for r in vocab_rows]
ax1.bar(names_v, [r["symbols"] for r in vocab_rows],
        color=["tab:blue", "tab:orange", "tab:green"])
for i, r in enumerate(vocab_rows):
    ax1.text(i, r["symbols"], f"{r['symbols']} symbols\n{r['letters']} letters\n"
                              f"{len(r['digits'])} digits\n{len(r['sentence'])} sentence punct",
             ha="center", va="bottom", fontsize=8)
ax1.set(ylabel="pronounceable symbols",
        ylim=(0, max(r["symbols"] for r in vocab_rows) * 1.5),
        title="Every MMS voice is a character vocabulary of about 40 symbols")
ax2.bar(names_v, [r["n_dropped"] for r in vocab_rows],
        color=["tab:blue", "tab:orange", "tab:green"])
for i, r in enumerate(vocab_rows):
    ax2.text(i, r["n_dropped"], str(r["n_dropped"]), ha="center", va="bottom")
ax2.set(ylabel="characters deleted", title="Characters silently deleted from our own\n"
                                           "pipeline output, before the model saw it")
plt.tight_layout(); plt.show()

In [ ]:
# Latin labels only in the figure above: DejaVu Sans renders Arabic unshaped and left to right,
# so plotting the dropped glyphs would misrepresent the script it is complaining about. The
# glyphs are printed instead, where the notebook can render them properly.
print(arabic)
ipd.Audio(synthesise(arabic, TTS_ARA), rate=SAMPLING_RATE)

**Not one of these voices can say a full stop, a comma, an exclamation mark or a question mark.**
Punctuation is not mispronounced by these models; it is removed, and every prosodic cue a reader
would take from it goes with it. English keeps digits 0 to 6 and stops there, so even in English
"1987" is only partly sayable and "1989" is not. Arabic is the extreme case: 38 symbols, 35 of them
bare letters, no digits, no sentence punctuation, not even an apostrophe or a hyphen, and no harakat.

This matters for the hands-on, because `space/` targets Arabic. The grader only asks *"is this
confidently not English"*, and deleted punctuation cannot make it fail. Your Space passes while
quietly dropping characters — this unit's thesis in one line: **the seam was lossy and the score did
not notice.**

## That's Unit 7

**The grading Space is currently down.** `huggingface-course/audio-course-u7-assessment` is in
`RUNTIME_ERROR`: it loads `facebook/mms-lid-126` (3.86 GB) at import and dies on *"No space left on
device"*. The separate progress Space 404s on a private dataset whose token has expired. Both are on
Hugging Face's side, so **this hands-on cannot be submitted right now** — `space/` in this folder is
prepared and ready to upload, not published. The unit `README.md` has the details.

Three things to carry forward:

- **`task="translate"` only ever produces English**, and `language=` alongside it is silently
  ignored. The hands-on requires non-English. Add a translation hop; do not force a language token.
- **The Hub's `pipeline_tag` is a hint, not a contract.** `pipeline("text-generation",
  model="MBZUAI/LaMini-Flan-T5-248M")` echoes your prompt back verbatim and never raises.
- **Check the null before you report a number.** A scorer that picks the better of two label
  mappings cannot go below 50%, so 50% is not the baseline and the difference decides whether you
  have a result or noise.

**Also in this folder**

- `walkthrough.py` — the same nine sections as a script, saving figures and clips
- `assistant.py` — the four-stage voice assistant (canned, or `--live` in Gradio)
- `meeting.py` — word timestamps, the merge, and the diarization sweep over every speaker pair
- `gradio_demo.py` — four tabs: translate, compare cascades, assistant, meeting
- `space/` — the hands-on Space, ready to upload

**Supplemental reading from the course**

- [Textless speech-to-speech translation on real data](https://ai.meta.com/research/publications/textless-speech-to-speech-translation-on-real-data/) — STST with discrete units
- [Translatotron 2](https://arxiv.org/abs/2107.08661) — direct speech-to-speech, no text bottleneck
- [Accurate detection of wake word start and end using a CNN](https://www.amazon.science/publications/accurate-detection-of-wake-word-start-and-end-using-a-cnn)
- [pyannote.audio 2.1](https://huggingface.co/pyannote/speaker-diarization) — the diarization section 7 could not run
- [WhisperX](https://arxiv.org/abs/2303.00747) — forced alignment for better word timestamps